# Clase 214 — Parquet vs CSV vs Avro: benchmarks y schema evolution

Requiere: `pip install pyarrow polars duckdb fastavro`.

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq, polars as pl, pandas as pd, numpy as np, time, json
from pathlib import Path
import tempfile, shutil

WORK = Path(tempfile.gettempdir()) / 'parquet_bench'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()

rng = np.random.default_rng(42)
N = 2_000_000
df = pl.DataFrame({
    'zone_id': rng.integers(0, 100, N),
    'fare':    rng.uniform(5, 100, N),
    'tip':     rng.uniform(0, 20, N),
    'pickup_date': pl.date(2024, 1, 1) + pl.duration(days=pl.Series(rng.integers(0, 90, N))),
    'borough': rng.choice(['Manhattan', 'Brooklyn', 'Queens', 'Bronx'], N),
    'note':    rng.choice(['short ride', 'long ride', 'airport', 'rush hour', None], N),
})
print(f'dataset: {N:,} rows, {len(df.columns)} cols')

## 1. Benchmark: tamaño por formato

In [ ]:
results = []
df.write_csv(WORK / 'd.csv')
results.append(('CSV', (WORK / 'd.csv').stat().st_size))

for comp in ['snappy', 'zstd', 'gzip', 'lz4']:
    p = WORK / f'd_{comp}.parquet'
    df.write_parquet(p, compression=comp)
    results.append((f'Parquet/{comp}', p.stat().st_size))

import gzip
j = WORK / 'd.json.gz'
with gzip.open(j, 'wt') as f:
    for row in df.iter_rows(named=True):
        f.write(json.dumps(row, default=str) + '\n')
results.append(('JSONL/gzip', j.stat().st_size))

print(f'{"formato":18} {"MB":>8}  ratio vs CSV')
csv_size = next(s for n, s in results if n == 'CSV')
for name, size in results:
    print(f'{name:18} {size / 1024 / 1024:>8.2f}  {size / csv_size:.2%}')

## 2. Benchmark: tiempo de query analítica

In [ ]:
import duckdb
con = duckdb.connect()

def bench(name, sql):
    t0 = time.perf_counter()
    n = con.execute(sql).fetchone()[0]
    return name, (time.perf_counter() - t0) * 1000, n

csv_p = str(WORK / 'd.csv').replace(chr(92), '/')
snap_p = str(WORK / 'd_snappy.parquet').replace(chr(92), '/')
zstd_p = str(WORK / 'd_zstd.parquet').replace(chr(92), '/')

queries = [
    ('CSV',  f"SELECT COUNT(*) FROM '{csv_p}' WHERE borough='Manhattan'"),
    ('Parquet snappy', f"SELECT COUNT(*) FROM '{snap_p}' WHERE borough='Manhattan'"),
    ('Parquet zstd',   f"SELECT COUNT(*) FROM '{zstd_p}' WHERE borough='Manhattan'"),
]
for name, sql in queries:
    n, t, _ = bench(name, sql)
    print(f'{n:20} {t:>8.1f} ms')

# Column pruning: leer solo 1 columna
print('\n--- SELECT 1 column con filter ---')
for name, p in [('CSV', csv_p), ('Parquet snappy', snap_p)]:
    t0 = time.perf_counter()
    con.execute(f"SELECT AVG(fare) FROM '{p}' WHERE borough='Manhattan'").fetchone()
    print(f'{name:20} {(time.perf_counter() - t0) * 1000:>8.1f} ms')

## 3. Inspeccionar metadata Parquet

In [ ]:
meta = pq.ParquetFile(snap_p).metadata
print(f'archivo: {meta.num_rows:,} rows, {meta.num_row_groups} row groups, {meta.num_columns} columns')

rg = meta.row_group(0)
print(f'\nrow group 0: {rg.num_rows:,} rows')
for i in range(rg.num_columns):
    col = rg.column(i)
    s = col.statistics
    if s:
        print(f'  col {i} ({col.path_in_schema:12}): min={s.min}, max={s.max}, nulls={s.null_count}')

## 4. Avro: serialización + schema evolution

In [ ]:
import fastavro
from io import BytesIO

schema_v1 = {
    'type': 'record',
    'name': 'Click',
    'fields': [
        {'name': 'user_id', 'type': 'string'},
        {'name': 'page',    'type': 'string'},
        {'name': 'ts',      'type': 'double'},
    ],
}

schema_v2 = {  # backward compatible: agregar campo OPCIONAL con default
    'type': 'record',
    'name': 'Click',
    'fields': [
        {'name': 'user_id', 'type': 'string'},
        {'name': 'page',    'type': 'string'},
        {'name': 'ts',      'type': 'double'},
        {'name': 'session_id', 'type': ['null', 'string'], 'default': None},  # nuevo
    ],
}

events_v2 = [
    {'user_id': 'u1', 'page': '/foo', 'ts': 1.0, 'session_id': 's1'},
    {'user_id': 'u2', 'page': '/bar', 'ts': 2.0, 'session_id': None},
]

buf = BytesIO()
fastavro.writer(buf, schema_v2, events_v2)
data = buf.getvalue()
print(f'2 records v2: {len(data)} bytes')

# Consumer viejo (con schema v1) puede leer data v2 — los campos nuevos se ignoran
buf.seek(0)
reader = fastavro.reader(buf, reader_schema=schema_v1)
for r in reader:
    print('leído con schema v1:', r)
print('\n→ schema evolution backward-compatible OK')

## Ejercicio guiado

1. Bajá 1 año de NYC Taxi (CSV original). Convertí a Parquet con cada compresión. Reporte tamaño + tiempo de query.
2. Sobre el Parquet final, particioná por `pickup_date` (`partitionBy`). Compará tiempo de `WHERE pickup_date='X'` con/sin particionado.
3. Hacé un cambio NO compatible en Avro (renombrar `page` → `url` sin alias). Confirmá que rompe el consumer viejo.
4. Investigá Delta Lake o Iceberg sobre tu Parquet — agregá ACID + time travel.
5. Estimá ahorro de S3 al migrar CSV → Parquet zstd en tu workload real.

## Conclusiones

- Parquet es 5-20× más chico que CSV y 10-100× más rápido en queries analíticas (column pruning + predicate pushdown).
- Compresión zstd domina ratio; snappy domina speed.
- Avro brilla en streaming + schema evolution; Parquet brilla en storage + queries.
- CSV solo sobrevive para interop humano y datasets triviales.